# PROJECT -3. BOOKING

Первоначальная версия датасета содержит 17 полей со следующей информацией:

hotel_address — адрес отеля;
review_date — дата, когда рецензент разместил соответствующий отзыв;
average_score — средний балл отеля, рассчитанный на основе последнего комментария за последний год;
hotel_name — название отеля;
reviewer_nationality — страна рецензента;
negative_review — отрицательный отзыв, который рецензент дал отелю;
review_total_negative_word_counts — общее количество слов в отрицательном отзыв;
positive_review — положительный отзыв, который рецензент дал отелю;
review_total_positive_word_counts — общее количество слов в положительном отзыве.
reviewer_score — оценка, которую рецензент поставил отелю на основе своего опыта;
total_number_of_reviews_reviewer_has_given — количество отзывов, которые рецензенты дали в прошлом;
total_number_of_reviews — общее количество действительных отзывов об отеле;
tags — теги, которые рецензент дал отелю;
days_since_review — количество дней между датой проверки и датой очистки;
additional_number_of_scoring — есть также некоторые гости, которые просто поставили оценку сервису, но не оставили отзыв. Это число указывает, сколько там действительных оценок без проверки.
lat — географическая широта отеля;
lng — географическая долгота отеля.

In [85]:
#ЗАГРУЗИМ ДАННЫЕ

import pandas as pd

hotels = pd.read_csv('hotels.csv')
hotels.head(2)

,hotel_address,additional_number_of_scoring,review_date,average_score,hotel_name,reviewer_nationality,negative_review,review_total_negative_word_counts,total_number_of_reviews,positive_review,review_total_positive_word_counts,total_number_of_reviews_reviewer_has_given,reviewer_score,tags,days_since_review,lat,lng
0,Stratton Street Mayfair Westminster Borough Lo...,581,2/19/2016,8.4,The May Fair Hotel,United Kingdom,Leaving,3,1994,Staff were amazing,4,7,10.0,"[' Leisure trip ', ' Couple ', ' Studio Suite ...",531 day,51.507894,-0.143671
1,130 134 Southampton Row Camden London WC1B 5AF...,299,1/12/2017,8.3,Mercure London Bloomsbury Hotel,United Kingdom,poor breakfast,3,1361,location,2,14,6.3,"[' Business trip ', ' Couple ', ' Standard Dou...",203 day,51.521009,-0.123097


In [86]:
# В каких столбцах данные содержат пропущенные значения?
#В каких столбцах данные хранятся в числовом формате?
hotels.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 386803 entries, 0 to 386802
Data columns (total 17 columns):
 #   Column                                      Non-Null Count   Dtype  
---  ------                                      --------------   -----  
 0   hotel_address                               386803 non-null  object 
 1   additional_number_of_scoring                386803 non-null  int64  
 2   review_date                                 386803 non-null  object 
 3   average_score                               386803 non-null  float64
 4   hotel_name                                  386803 non-null  object 
 5   reviewer_nationality                        386803 non-null  object 
 6   negative_review                             386803 non-null  object 
 7   review_total_negative_word_counts           386803 non-null  int64  
 8   total_number_of_reviews                     386803 non-null  int64  
 9   positive_review                             386803 non-null  object 
 

Задачу, которая стоит перед вами, можно свести к пяти пунктам:

Удаление строковых значений. Вам необходимо удалить из набора данных столбцы, данные в которых представлены не числами.
Очистка от пропущенных значений. На предыдущем шаге мы делали это самым грубым из всех возможных способов, сейчас попробуйте подойти к процессу более гибко.
Создание новых признаков. Мы попробуем создать новые столбцы с данными из существующих данных или с использованием внешних источников.
Преобразование признаков. Применим различные преобразования над признаками вроде нормализации, стандартизации.
Отбор признаков. Используем анализ мультиколлинеарности как шаг отбора признаков для модели.

In [87]:
# Сколько уникальных названий отелей представлено в наборе данных?
hotels['hotel_name'].nunique()

1492

In [88]:
hotels['review_date'] = pd.to_datetime(hotels['review_date'], dayfirst=False)

In [89]:
# Когда был оставлен самый свежий отзыв? Введите ответ в формате yyyy-mm-dd.
# Когда был оставлен самый первый отзыв? Введите ответ в формате yyyy-mm-dd.
display(hotels['review_date'].max())
display(hotels['review_date'].min())                         

Timestamp('2017-08-03 00:00:00')

Timestamp('2015-08-04 00:00:00')

In [239]:
# Сколько уникальных тегов представлено в наборе данных?


6

In [240]:
#посчитаем количество тегов (это кол-во запятых между ними+1)
hotels['tags_num'] = hotels['tags'].str.count(',')+1
hotels['tags_num'].describe()

count    386803.000000
mean          4.569566
std           0.520163
min           1.000000
25%           4.000000
50%           5.000000
75%           5.000000
max           6.000000
Name: tags_num, dtype: float64

In [389]:
#обработаем теги
new_tags = hotels['tags'].str.replace(" ']", '')
new_tags = new_tags.str.replace("[' ", '')
new_tags = new_tags.str.replace("' ", '')
new_tags = new_tags.str.replace(" '", '')


tags_a= pd.DataFrame()
tags_a['tags'] = new_tags.astype(str)

#с животными
tags_a['pet'] = tags_a['tags'].apply(lambda x: x.split(', ')[0] if x.split(', ')[0] == 'With a pet' else 'No pet')
tags_a['cleaned_tags'] = tags_a['tags'].str.replace('With a pet, ', '')
#с мобильного
tags_a['device'] = tags_a['tags'].apply(lambda x: x.split(', ')[-1] if x.split(', ')[-1] == 'Submitted from a mobile device' else 'other')
tags_a['cleaned_tags'] = tags_a['cleaned_tags'].str.replace(', Submitted from a mobile device', '')

#команировка/отдых
tags_a['trip_type'] = tags_a['cleaned_tags'].apply(lambda x: x.split(', ')[0] if x.split(', ')[0] == 'Leisure trip' or x.split(', ')[0] == 'Business trip'  else 'other')
tags_a['cleaned_tags'] = tags_a['cleaned_tags'].str.replace('Leisure trip, ', '')
tags_a['cleaned_tags'] = tags_a['cleaned_tags'].str.replace('Business trip, ', '')

#компания
tags_a['traveler'] = tags_a['cleaned_tags'].apply(lambda x: x.split(', ')[0])
list_of_rep = ['Couple,', 'Solo traveler', 'Family with young children', 'Group',
       'Family with older children', 'Travelers with friends']

tags_a['cleaned_tags'] = tags_a['cleaned_tags'].replace(list_of_rep, '')




#срок пребывания (в строке попались некоторые ошибки, которые можно вычистить)
tags_a['duration_of_stay'] = tags_a['cleaned_tags'].apply(lambda x: x.split(', ')[-1])
#чтобы вычистить ошибки, очистим инфорацию и оставим только числовое значение кол-ва ночей
tags_a['number_of_days'] = tags_a['duration_of_stay'].str.extract('(\d+)', expand=False)


#добавим информацию о кол-ве тегов
tags_a['tags_num'] = hotels['tags'].str.count(',')+1

tags_a


,tags,pet,cleaned_tags,device,trip_type,traveler,tags_num,duration_of_stay
0,"Leisure trip, Couple, Studio Suite, Stayed 2 n...",No pet,"Couple, Studio Suite, Stayed 2 nights",Submitted from a mobile device,Leisure trip,Couple,5,Stayed 2 nights
1,"Business trip, Couple, Standard Double Room, S...",No pet,"Couple, Standard Double Room, Stayed 1 night",other,Business trip,Couple,4,Stayed 1 night
2,"Leisure trip, Solo traveler, Modern Double Roo...",No pet,"Solo traveler, Modern Double Room Echo, Stayed...",Submitted from a mobile device,Leisure trip,Solo traveler,5,Stayed 3 nights
3,"Leisure trip, Solo traveler, Standard Room wit...",No pet,"Solo traveler, Standard Room with 1 Double Bed...",other,Leisure trip,Solo traveler,4,Stayed 1 night
4,"Business trip, Couple, Standard Double or Twin...",No pet,"Couple, Standard Double or Twin Room, Stayed 6...",other,Business trip,Couple,4,Stayed 6 nights
...,...,...,...,...,...,...,...,...
386798,"Leisure trip, Group, Club Double or Twin Room,...",No pet,"Group, Club Double or Twin Room, Stayed 2 nights",Submitted from a mobile device,Leisure trip,Group,5,Stayed 2 nights
386799,"Leisure trip, Couple, Standard Double Room, St...",No pet,"Couple, Standard Double Room, Stayed 4 nights",Submitted from a mobile device,Leisure trip,Couple,5,Stayed 4 nights
386800,"Business trip, Solo traveler, Single Room, Sta...",No pet,"Solo traveler, Single Room, Stayed 1 night",other,Business trip,Solo traveler,4,Stayed 1 night
386801,"Leisure trip, Solo traveler, Deluxe Double Roo...",No pet,"Solo traveler, Deluxe Double Room, Stayed 2 ni...",other,Leisure trip,Solo traveler,4,Stayed 2 nights


In [385]:
tags_a[tags_a['tags_num']==2]['cleaned_tags']


1865            
25450           
33053     Couple
54240     Couple
56081           
           ...  
358001          
373516          
375091    Couple
382344          
384943    Couple
Name: cleaned_tags, Length: 65, dtype: object

In [378]:
list_of_rep = ['Couple', 'Solo traveler', 'Family with young children', 'Group',
       'Family with older children', 'Travelers with friends']

tags_a['cleaned_tags'] = tags_a['cleaned_tags'].replace(list_of_rep, '')


tags_a['cleaned_tags'] = tags_a['cleaned_tags'].apply(lambda word: for word in list_of_rep  replace('word', '') )

In [393]:
tags_a['duration_of_stay'].unique()

tags_a['number_of_days'] = tags_a['duration_of_stay'].str.extract('(\d+)', expand=False)

tags_a['number_of_days'] 

0         2
1         1
2         3
3         1
4         6
         ..
386798    2
386799    4
386800    1
386801    2
386802    4
Name: number_of_days, Length: 386803, dtype: object

In [396]:
tags_a[tags_a['duration_of_stay']== 'Stayed 2 nights']

,tags,pet,cleaned_tags,device,trip_type,traveler,tags_num,duration_of_stay,number_of_days
0,"Leisure trip, Couple, Studio Suite, Stayed 2 n...",No pet,"Couple, Studio Suite, Stayed 2 nights",Submitted from a mobile device,Leisure trip,Couple,5,Stayed 2 nights,2
17,"Leisure trip, Group, Standard Room, Stayed 2 n...",No pet,"Group, Standard Room, Stayed 2 nights",Submitted from a mobile device,Leisure trip,Group,5,Stayed 2 nights,2
18,"Leisure trip, Solo traveler, Standard Double R...",No pet,"Solo traveler, Standard Double Room, Stayed 2 ...",other,Leisure trip,Solo traveler,4,Stayed 2 nights,2
21,"Business trip, Solo traveler, Executive Double...",No pet,"Solo traveler, Executive Double Room, Stayed 2...",other,Business trip,Solo traveler,4,Stayed 2 nights,2
25,"Leisure trip, Couple, X Ordinary Room, Stayed ...",No pet,"Couple, X Ordinary Room, Stayed 2 nights",other,Leisure trip,Couple,4,Stayed 2 nights,2
...,...,...,...,...,...,...,...,...,...
386786,"Leisure trip, Couple, Superior Double Room, St...",No pet,"Couple, Superior Double Room, Stayed 2 nights",Submitted from a mobile device,Leisure trip,Couple,5,Stayed 2 nights,2
386791,"Business trip, Solo traveler, Superior King Ro...",No pet,"Solo traveler, Superior King Room, Stayed 2 ni...",other,Business trip,Solo traveler,4,Stayed 2 nights,2
386793,"Leisure trip, Group, Junior Suite, Stayed 2 ni...",No pet,"Group, Junior Suite, Stayed 2 nights",Submitted from a mobile device,Leisure trip,Group,5,Stayed 2 nights,2
386798,"Leisure trip, Group, Club Double or Twin Room,...",No pet,"Group, Club Double or Twin Room, Stayed 2 nights",Submitted from a mobile device,Leisure trip,Group,5,Stayed 2 nights,2


In [391]:
tags_a.groupby('duration_of_stay')['tags'].count()

duration_of_stay
                        84
Couple                  62
Stayed 1 night      145373
Stayed 10 nights       663
Stayed 11 nights       306
Stayed 12 nights       217
Stayed 13 nights       174
Stayed 14 nights       184
Stayed 15 nights        87
Stayed 16 nights        38
Stayed 17 nights        27
Stayed 18 nights        24
Stayed 19 nights        23
Stayed 2 nights     100263
Stayed 20 nights        17
Stayed 21 nights        19
Stayed 22 nights         8
Stayed 23 nights         6
Stayed 24 nights         5
Stayed 25 nights         4
Stayed 26 nights         6
Stayed 27 nights        10
Stayed 28 nights         7
Stayed 29 nights         3
Stayed 3 nights      72000
Stayed 30 nights        10
Stayed 4 nights      35748
Stayed 5 nights      15611
Stayed 6 nights       7399
Stayed 7 nights       5549
Stayed 8 nights       1910
Stayed 9 nights        966
Name: tags, dtype: int64

In [321]:
tags_a.count()

tags                386803
pet                 386803
device              230778
trip_type           374614
duration_of_stay    156025
dtype: int64

In [235]:
new_tags[new_tags['device']!='Submitted from a mobile device']['duration_of_stay'].unique()

array(['Submitted from a mobile device', None, 'Stayed 2 nights',
       'Stayed 1 night', 'Stayed 3 nights', 'Stayed 5 nights',
       'Stayed 7 nights', 'Stayed 4 nights', 'Stayed 6 nights',
       'Stayed 9 nights'], dtype=object)

In [ ]:
# На вход данной функции поступает строка с адресом.
def get_tags(tags):

    new_tags = tags.replace(" ']", '')
    new_tags = new_tags.replace("[' ", '')
    new_tags = new_tags.replace("' ", '')
    new_tags = new_tags.replace(" '", '')
    
    tags_list = new_tags.split(', ')[0]
    
    if tags_list[0] == 'With a pet'
        pet = tags_list[0]
    if tags_list[0] != 'With a pet' and tags_list[0] =='Leisure trip' or tags_list[0] =='Business trip'
        pet = None
        trip_type = tags_list[0]
    if tags_list[-1] == 'Submitted from a mobile device'
        device = tags_list[-1]
    if tags_list[-1] != 'Submitted from a mobile device'
        duration_of_stay = tags_list[-1]
    else: none
        
    
    
    
    

In [236]:
hotels[['pet','trip_type', 'duration','device']] = hotels['tags'].apply(get_tags),expand=True
hotels

ValueError: Columns must be same length as key

In [220]:
new_tags[new_tags['comma_count']==5]['pet'].unique()

array(['With a pet'], dtype=object)

In [ ]:
# На вход данной функции поступает строка с адресом.
def get_tags(tags):
# Метод split() разбивает строку на слова по пробелу.
# В результате получаем список слов в строке и заносим его в переменную address_list.
    comma_count = tags.count(',')
    tags_list = tags.split(', ')
# Обрезаем список, оставляя в нём только последний элемент,
# потенциальный подтип улицы, и заносим в переменную street_type.
    if comma_count == 5:
        pet = tags_list[0].replace('[^\w\s]','')
        trip_type = tags_list[1].replace('[^\w\s]','')
        traveler = tags_list[2].replace('[^\w\s]','')
        room = tags_list[3].replace('[^\w\s]','')
        duration = tags_list[4].replace('[^\w\s]','')
        device = tags_list[5].replace('[^\w\s]','')
        return pet,trip_type,traveler,room,duration,device
    if comma_count == 4:
        trip_type = tags_list[0].replace('[^\w\s]','')
        traveler = tags_list[1].replace('[^\w\s]','')
        room = tags_list[2].replace('[^\w\s]','')
        duration = tags_list[3].replace('[^\w\s]','')
        device = tags_list[5].replace('[^\w\s]','')
        return trip_type,traveler,room,duration,device
    if comma_count == 3:
        trip_type = tags_list[0].replace('[^\w\s]','')
        traveler = tags_list[1].replace('[^\w\s]','')
        room = tags_list[2].replace('[^\w\s]','')
        return trip_type,traveler,room
    if comma_count == 2:
        trip_type = tags_list[0].replace('[^\w\s]','')
        traveler = tags_list[1].replace('[^\w\s]','')
        return trip_type,traveler
    if comma_count == 1:
        trip_type = tags_list[0].replace('[^\w\s]','')
        return trip_type
    else: None

In [194]:
new_tags['duration_of_stay'].unique()

array(['Stayed 2 nights', 'Stayed 1 night', 'Stayed 3 nights',
       'Stayed 6 nights', 'Stayed 4 nights',
       'Submitted from a mobile device', 'Stayed 5 nights', None,
       'Stayed 8 nights', 'Stayed 7 nights', 'Stayed 10 nights',
       'Stayed 14 nights', 'Stayed 19 nights', 'Standard Double Room',
       'Stayed 13 nights', 'Stayed 9 nights', 'Superior Twin Room',
       'Family Room', 'Stayed 17 nights', 'Stayed 11 nights',
       'Superior Double Room', 'King Room with City View Top floors',
       '2 rooms', 'Double or Twin Room',
       'Executive Double Room Eiffel Studio D Artiste with view of Paris',
       'Deluxe King Room', 'King Room', 'Superior Room',
       'Stayed 12 nights', 'Stayed 15 nights', 'Stayed 21 nights',
       'Stayed 18 nights', 'Double or Twin Room with View',
       'Superior Double or Twin Room with View', 'Suite', 'Double Room',
       'Stayed 16 nights', 'Executive Twin Room', 'Classic Queen Room',
       'Queen Guest Room', 'Standard Room',
 

In [193]:
new_tags['duration_of_stay'].nunique()

278

In [147]:
# На вход данной функции поступает строка с адресом.
def get_tags(tags):
# Метод split() разбивает строку на слова по пробелу.
# В результате получаем список слов в строке и заносим его в переменную address_list.
    comma_count = tags.count(',')
    tags_list = tags.split(', ')
# Обрезаем список, оставляя в нём только последний элемент,
# потенциальный подтип улицы, и заносим в переменную street_type.
    if comma_count == 5:
        pet = tags_list[0].replace('[^\w\s]','')
        trip_type = tags_list[1].replace('[^\w\s]','')
        traveler = tags_list[2].replace('[^\w\s]','')
        room = tags_list[3].replace('[^\w\s]','')
        duration = tags_list[4].replace('[^\w\s]','')
        device = tags_list[5].replace('[^\w\s]','')
        return pet,trip_type,traveler,room,duration,device
    if comma_count == 4:
        trip_type = tags_list[0].replace('[^\w\s]','')
        traveler = tags_list[1].replace('[^\w\s]','')
        room = tags_list[2].replace('[^\w\s]','')
        duration = tags_list[3].replace('[^\w\s]','')
        device = tags_list[5].replace('[^\w\s]','')
        return trip_type,traveler,room,duration,device
    if comma_count == 3:
        trip_type = tags_list[0].replace('[^\w\s]','')
        traveler = tags_list[1].replace('[^\w\s]','')
        room = tags_list[2].replace('[^\w\s]','')
        return trip_type,traveler,room
    if comma_count == 2:
        trip_type = tags_list[0].replace('[^\w\s]','')
        traveler = tags_list[1].replace('[^\w\s]','')
        return trip_type,traveler
    if comma_count == 1:
        trip_type = tags_list[0].replace('[^\w\s]','')
        return trip_type
    else: None

In [148]:
hotels[['trip_type', 'traveler', 'room_type', 'duration','device']] = hotels['tags'].apply(get_tags)
hotels

ValueError: Columns must be same length as key

In [121]:
hotels['tags'][28549]

"[' Leisure trip ', ' Travelers with friends ', ' Double or Twin Room ', ' Stayed 4 nights ', ' Submitted from a mobile device ']"

In [ ]:
new_tags = hotels['tags'].split(', ',expand=True)
new_tags



In [123]:
new_tags[0].replace('[^\w\s]','')

0          [' Leisure trip '
1         [' Business trip '
2          [' Leisure trip '
3          [' Leisure trip '
4         [' Business trip '
                 ...        
386798     [' Leisure trip '
386799     [' Leisure trip '
386800    [' Business trip '
386801     [' Leisure trip '
386802     [' Leisure trip '
Name: 0, Length: 386803, dtype: object

In [ ]:
hotels[['trip_type', 'traveler', 'room_type', 'duration','device']]= pd.DataFrame(hotels['tags'].tolist(), index=hotels['tags'].index)


In [12]:
#Сколько уникальных тегов представлено в наборе данных?
hotels['tags'].nunique()

47135